# Notebook 12 — Error consistency vs. BFT factorization similarity (CIFAR-10, correlational)

**Question (purely correlational).** Across a set of CIFAR-10 CNNs, do model pairs whose BFT factorization at a layer is more similar also make *more consistent errors* on the class / sub-class that factor represents?

For every model we factorize each layer's weight×activation arbor (BFT's per-layer object) over one **shared** image set. For each model **pair** and each layer we (i) **Hungarian-match** the two factorizations and read the matched cosine similarity per factor, and (ii) measure **error consistency** (Cohen's κ, Geirhos/Meding/Wichmann 2020) on the images that factor represents — a *class* at the classifier layer, a *sub-class* in the conv layers (the paper's bear→species structure). Then we correlate factor-similarity vs. error-consistency across all (pair, layer, factor) cells.

**Models.** The 5 seed checkpoints (`cifar10_cnn_seed0..4`) are the same architecture and recipe, so they can be *too similar* (little spread in either axis). To widen the range we additionally train **two models with different recipes** (strong-augmentation/long and no-augmentation/short); these diverge in behavior from the seeds and from each other, which is what a correlation needs.

**No causal manipulation** — this notebook is observational only.

Run one experiment; `NB12_MODE=cluster` for the full run (GPU), `local` for a laptop smoke test. See the final cell for the cluster recipe. Portable outputs: `data/results/nb12_cnn_cifar.json` and the figure bundle `figures/figdata/nb12_error_consistency.{npz,json}`.

## §0 · Setup, mode & helpers

In [ ]:
import os, sys, json, copy, time, warnings
sys.path.insert(0, '..')
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets
import torchvision.transforms as T
from sklearn.decomposition import MiniBatchNMF
from scipy.stats import spearmanr, rankdata, pearsonr

from src import (SmallCNN, load_experiment, save_experiment, collect_layer_dicts,
                 get_cifar10_loaders, figdata)
from src.bft import compute_conv_joint_arbors, compute_joint_arbors_normalized
from src.training import train_epoch, evaluate
from src.robustness_utils import _hungarian_align

warnings.filterwarnings('ignore')
RNG  = np.random.default_rng(0)
REPO = os.path.abspath('..')
RES_DIR    = os.path.join(REPO, 'data', 'results')
MODEL_ROOT = os.path.join(REPO, 'data', 'models')
FIG_DIR    = os.path.join(REPO, 'figs', '12_error_consistency')
for d in (RES_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)

MODE   = 'cluster'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      ('mps' if torch.backends.mps.is_available() else 'cpu'))

# 'cluster' = full run; 'local' = laptop smoke test (small, NOT publication-grade)
if MODE == 'cluster':
    N_PER_CLASS, NMF_SUB, NMF_MAX_ITER, TOP_Q = 150, 1000, 300, 0.30
    TRAIN_EXTRA = True                 # train the 2 extra-recipe models
else:
    N_PER_CLASS, NMF_SUB, NMF_MAX_ITER, TOP_Q = 25, 200, 60, 0.40
    TRAIN_EXTRA = False                # too slow on CPU; smoke uses perturbed clones instead

LAYER_RANKS = [6, 6, 6, 6, 10]         # K per layer (4 conv + classifier); output≈classes
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
RESULT_PATH = os.path.join(RES_DIR, 'nb12_cnn_cifar.json')
print(f'MODE={MODE}  DEVICE={DEVICE}  N_PER_CLASS={N_PER_CLASS}  TRAIN_EXTRA={TRAIN_EXTRA}')


def jsonable(o):
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return float(o)
    return o if isinstance(o, (float, int, str, bool)) or o is None else str(o)

def dump_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as f: json.dump(jsonable(obj), f, indent=1)
    os.replace(tmp, path)

@torch.no_grad()
def predict_correct(model, loader):
    '''Per-sample correctness on the shared loader (loader order).'''
    corr = []
    model.eval()
    for x, y in loader:
        out = model(x.to(DEVICE))
        logits = out[0] if isinstance(out, tuple) else out
        corr.append((logits.argmax(1).cpu() == y).numpy())
    return np.concatenate(corr)

def error_consistency(a, b):
    '''Cohen kappa on trial-by-trial correctness (Geirhos/Meding/Wichmann 2020).'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 8: return np.nan
    c_obs = np.mean(a == b); pa, pb = a.mean(), b.mean()
    c_exp = pa*pb + (1-pa)*(1-pb)
    return np.nan if c_exp >= 1 - 1e-9 else (c_obs - c_exp) / (1 - c_exp)

def partial_spear(x, y, covs):
    x, y = np.asarray(x, float), np.asarray(y, float)
    xr, yr = rankdata(x), rankdata(y)
    Z = np.column_stack([rankdata(np.asarray(c, float)) for c in covs] + [np.ones_like(xr)])
    bx = np.linalg.lstsq(Z, xr, rcond=None)[0]; by = np.linalg.lstsq(Z, yr, rcond=None)[0]
    return pearsonr(xr - Z@bx, yr - Z@by)

def perm_null(x, y, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    obs = spearmanr(x, y).statistic
    null = np.array([spearmanr(rng.permutation(x), y).statistic for _ in range(n)])
    return float(obs), float(null.mean()), float(null.std()), float((np.sum(null >= obs)+1)/(n+1))

## §1 · Shared evaluation set

In [ ]:
tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
test_ds = datasets.CIFAR10(os.path.join(REPO, 'data'), train=False, download=True, transform=tf)
tt = np.array(test_ds.targets)
shared_idx = np.concatenate([np.where(tt == c)[0][:N_PER_CLASS] for c in range(10)])
shared_loader = DataLoader(Subset(test_ds, shared_idx.tolist()), batch_size=256, shuffle=False)
shared_labels = tt[shared_idx]
N = len(shared_idx)
CLASS_NAMES = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
print(f'shared eval set: {N} images, {N_PER_CLASS}/class')

## §2 · Assemble the model roster

The 5 seed checkpoints, plus (on the cluster) two extra-recipe models trained here for spread. On a laptop with only `seed0`, the roster is padded with lightly-perturbed clones **purely to smoke-test the pipeline** (flagged `dummy`; not for results).

In [ ]:
def train_extra(name, augment, epochs, wd, lr=1e-3, channels=(32,64,128,256), seed=0):
    '''Train a SmallCNN with a given recipe and save it as a checkpoint (once).'''
    ed = os.path.join(MODEL_ROOT, f'{name}_seed{seed}')
    if os.path.exists(os.path.join(ed, 'weights.pt')):
        return load_experiment(ed, DEVICE)[0], name
    torch.manual_seed(seed); np.random.seed(seed)
    tr, te = get_cifar10_loaders(batch_size=128, root=os.path.join(REPO, 'data'), augment=augment)
    m = SmallCNN(channels=tuple(channels), n_classes=10, global_pool=True).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        loss, acc = train_epoch(m, tr, opt, crit, DEVICE)
        if (ep+1) % 10 == 0 or ep == epochs-1:
            print(f'    {name} epoch {ep+1}/{epochs}: train acc={acc:.3f}')
    _, ta = evaluate(m, te, crit, DEVICE)
    save_experiment(m, dict(arch='SmallCNN',
        arch_kwargs=dict(channels=list(channels), fc_dim=128, n_classes=10, global_pool=True),
        dataset='CIFAR10', dataset_kwargs=dict(root='../data/', batch_size=128, augment=augment),
        label_transform='identity', description=f'CIFAR SmallCNN recipe={name} aug={augment} ep={epochs} wd={wd}'), ed)
    print(f'  trained {name}: test acc={ta:.3f} -> {ed}')
    return m, name

def perturb_clone(model, scale, seed):
    g = torch.Generator().manual_seed(seed)
    m = copy.deepcopy(model).to(DEVICE).eval()
    with torch.no_grad():
        for p in m.parameters():
            p.add_(scale * (p.std() + 1e-6) * torch.randn(p.shape, generator=g).to(p.device))
    return m

roster = []   # list of (tag, model, is_dummy)
for sd in range(5):
    ed = os.path.join(MODEL_ROOT, f'cifar10_cnn_seed{sd}')
    if os.path.exists(os.path.join(ed, 'weights.pt')):
        roster.append((f'seed{sd}', load_experiment(ed, DEVICE)[0], False))
print(f'loaded {len(roster)} seed checkpoint(s)')

if TRAIN_EXTRA:
    EXTRA = [dict(name='cifar10_cnn_strongaug', augment='strong', epochs=60, wd=5e-4),
             dict(name='cifar10_cnn_noaug_short', augment='none', epochs=20, wd=0.0)]
    for e in EXTRA:
        m, nm = train_extra(**e)
        roster.append((nm.replace('cifar10_cnn_', ''), m, False))

# laptop smoke fallback: need >=3 models to exercise the pairwise + correlation code
if len(roster) < 3:
    base = roster[0][1]
    for k, sc in enumerate([0.04, 0.08, 0.12][:3 - len(roster) + 1]):
        roster.append((f'dummy{k}(σ={sc})', perturb_clone(base, sc, 100 + k), True))
    print('  [SMOKE] padded roster with perturbed clones — NOT for results')

# accuracy on the shared set
accs = {}
for tag, m, dum in roster:
    accs[tag] = float(predict_correct(m, shared_loader).mean())
    print(f'  {tag:26s} shared-set acc={accs[tag]:.3f}{"  [dummy]" if dum else ""}')
M = len(roster)
print(f'roster: {M} models')

## §3 · Per-model, per-layer factorizations

For each model and each layer we take BFT's per-layer object — the L2-normalized weight×activation joint arbor over the shared images (uniform stimulus weights, so the factorization spans all classes rather than one traced path) — and factorize its positive part with NMF at the layer's rank. `W_ℓ` (N×K) holds the per-image loadings; these are row-aligned across models (same shared images), so factors can be Hungarian-matched and their image sets compared directly.

In [ ]:
def layer_factorization(ld, K, nmf_seed=0):
    '''(N,K) NMF loadings of one layer positive joint arbor (uniform weights).'''
    if ld['type'] == 'conv':
        J = compute_conv_joint_arbors(ld['weight'], ld['input_fmap'],
                                      stimulus_weights=None, pool_method='avg')
    else:
        x = ld['input_fmap'].reshape(len(ld['input_fmap']), -1)
        J = compute_joint_arbors_normalized(ld['weight'], x, stimulus_weights=None)
    Jp = np.clip(J, 0, None).astype(np.float32); del J
    m = MiniBatchNMF(n_components=K, random_state=nmf_seed, max_iter=NMF_MAX_ITER,
                     batch_size=1024, init='random')
    sub = Jp if Jp.shape[0] <= NMF_SUB else Jp[RNG.choice(Jp.shape[0], NMF_SUB, replace=False)]
    m.fit(sub)
    W = m.transform(Jp).astype(np.float32); del Jp
    return W

# W_by_model[i][layer] = (N, K_layer) loadings ; correct_by_model[i] = (N,) bool
W_by_model, correct_by_model = [], []
t0 = time.time()
for i, (tag, model, dum) in enumerate(roster):
    coll = collect_layer_dicts(model, shared_loader, device=DEVICE, only_correct=False)
    lds = coll['layer_data']
    # sanity: collected order must match the shared loader (only_correct=False keeps all)
    assert len(coll['targets']) == N, f'{tag}: got {len(coll["targets"])} != {N}'
    Ws = [layer_factorization(ld, LAYER_RANKS[l]) for l, ld in enumerate(lds)]
    W_by_model.append(Ws)
    correct_by_model.append(predict_correct(model, shared_loader))
    print(f'  [{i+1}/{M}] {tag}: layers={[w.shape for w in Ws]}  ({time.time()-t0:.0f}s)')
n_layers = len(W_by_model[0])
print(f'factorized {M} models × {n_layers} layers in {time.time()-t0:.0f}s')

## §4 · Pairwise: Hungarian factor matching + per-factor error consistency

For every model pair and layer we Hungarian-match the two loading matrices (cosine over the shared images), then for each matched factor: its **similarity** = matched cosine, its **image set** = the top-`TOP_Q` images by the two models' averaged loading (the class/sub-class the factor represents), and its **error consistency** = Cohen's κ between the two models restricted to that set. Output-layer factors are annotated with their dominant class.

In [ ]:
def col_norm(A):
    return A / (np.linalg.norm(A, axis=0, keepdims=True) + 1e-12)

records = []
for i in range(M):
    for j in range(i + 1, M):
        ci, cj = correct_by_model[i], correct_by_model[j]
        for l in range(n_layers):
            WA, WB = W_by_model[i][l], W_by_model[j][l]
            S, perm, _ = _hungarian_align(WA, WB)          # cosine over shared images
            WAn, WBn = col_norm(WA), col_norm(WB)
            is_out = (l == n_layers - 1)
            for k in range(WA.shape[1]):
                sim = float(S[k, perm[k]])
                g = WAn[:, k] + WBn[:, perm[k]]            # combined loading
                thr = np.quantile(g, 1 - TOP_Q)
                I = np.where(g >= thr)[0]
                kap = error_consistency(ci[I], cj[I])
                pa, pb = float(ci[I].mean()), float(cj[I].mean())
                dom = int(np.bincount(shared_labels[I], minlength=10).argmax()) if is_out else -1
                records.append(dict(i=i, j=j, layer=l, factor=k, is_output=int(is_out),
                    sim=sim, kappa=kap, n_img=int(len(I)), acc_a=pa, acc_b=pb,
                    c_exp=pa*pb + (1-pa)*(1-pb), dom_class=dom))
print(f'{len(records)} (pair,layer,factor) cells')

## §5 · Correlational analysis

In [ ]:
import pandas as pd
df = pd.DataFrame(records)
d = df.dropna(subset=['kappa', 'sim']).copy()
print(f'{len(d)} valid cells | sim {d.sim.min():.2f}-{d.sim.max():.2f} | kappa {d.kappa.min():.2f}-{d.kappa.max():.2f}')

def report(sub, name):
    if len(sub) < 8 or sub.sim.std() < 1e-6:
        print(f'  {name:16s}: n={len(sub)} — too few / no spread'); return dict(n=int(len(sub)))
    rho, p = spearmanr(sub.sim, sub.kappa)
    obs, nm, ns, pp = perm_null(sub.sim.values, sub.kappa.values, n=2000)
    pr = partial_spear(sub.sim.values, sub.kappa.values, [sub.c_exp.values])
    print(f'  {name:16s}: n={len(sub):4d}  ρ={rho:+.3f} (perm p={pp:.4f})  partial|c_exp={pr[0]:+.3f}')
    return dict(n=int(len(sub)), rho=float(rho), perm_p=pp, partial_cexp=float(pr[0]),
                p_partial=float(pr[1]), sim_mean=float(sub.sim.mean()), kappa_mean=float(sub.kappa.mean()))

print('Spearman(factor similarity, error consistency):')
stats = {'all': report(d, 'all cells'),
         'output_classes': report(d[d.is_output == 1], 'output (classes)'),
         'conv_subclasses': report(d[d.is_output == 0], 'conv (subclasses)')}
per_layer = {}
for l in sorted(d.layer.unique()):
    per_layer[int(l)] = report(d[d.layer == l], f'layer {l}')
stats['per_layer'] = per_layer

## §6 · Portable outputs — results JSON + figure bundle

In [ ]:
# (a) full results JSON (everything needed to recompute / audit)
result = dict(experiment='cnn_cifar', mode=MODE, n_models=M,
              model_tags=[t for t, _, _ in roster], model_accs=accs,
              dummies=[t for t, _, dum in roster if dum],
              n_per_class=N_PER_CLASS, layer_ranks=LAYER_RANKS, top_q=TOP_Q,
              class_names=CLASS_NAMES, stats=stats, records=records)
dump_json(result, RESULT_PATH)
print('wrote', os.path.relpath(RESULT_PATH, REPO), '|', len(records), 'records')

# (b) figure bundle (plain arrays -> a paper figure builds off-cluster)
bundle = dict(
    sim=d.sim.to_numpy(np.float32), kappa=d.kappa.to_numpy(np.float32),
    layer=d.layer.to_numpy(np.int16), is_output=d.is_output.to_numpy(np.int16),
    c_exp=d.c_exp.to_numpy(np.float32), dom_class=d.dom_class.to_numpy(np.int16),
    per_layer_rho=np.array([per_layer[l].get('rho', np.nan) for l in sorted(per_layer)], np.float32),
    per_layer_ids=np.array(sorted(per_layer), np.int16),
    rho_all=float(stats['all'].get('rho', np.nan)),
    perm_p_all=float(stats['all'].get('perm_p', np.nan)),
    rho_output=float(stats['output_classes'].get('rho', np.nan)),
    rho_conv=float(stats['conv_subclasses'].get('rho', np.nan)),
    n_models=M, mode=MODE, model_tags=[t for t, _, _ in roster],
    n_per_class=N_PER_CLASS, note='CIFAR-10 correlational: BFT factor similarity vs error consistency')
figdata.save('nb12_error_consistency', bundle)

## §7 · Preview figure (scatter)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sc = ax[0].scatter(d.sim, d.kappa, c=d.layer, cmap='viridis', s=16, alpha=0.6)
if d.sim.std() > 1e-6:
    b1, b0 = np.polyfit(d.sim, d.kappa, 1); xx = np.linspace(d.sim.min(), d.sim.max(), 40)
    ax[0].plot(xx, b0 + b1*xx, 'k--', lw=1.4)
ax[0].set_xlabel('Hungarian-matched factor similarity'); ax[0].set_ylabel("error consistency κ")
ax[0].set_title(f"all cells: ρ={stats['all'].get('rho', float('nan')):+.2f}")
fig.colorbar(sc, ax=ax[0], label='layer (0=input … out)')
lids = sorted(per_layer); rhos = [per_layer[l].get('rho', np.nan) for l in lids]
ax[1].bar([str(l) for l in lids], rhos, color='#3b7dd8')
ax[1].axhline(0, color='k', lw=0.8); ax[1].set_xlabel('layer'); ax[1].set_ylabel('ρ(sim, κ)')
ax[1].set_title('per-layer correlation')
fig.tight_layout()
p = os.path.join(FIG_DIR, 'nb12_scatter.png'); fig.savefig(p, dpi=140, bbox_inches='tight')
print('saved', os.path.relpath(p, REPO)); plt.show()

## §8 · How to run on the cluster (GPU)

```bash
cd notebooks
NB12_MODE=cluster ../.venv/bin/python -m nbconvert --to notebook --execute \
  --output executed_12_error_consistency_cifar.ipynb \
  --ExecutePreprocessor.timeout=100000 12_error_consistency_cifar.ipynb
```

**Prerequisites:** `data/models/cifar10_cnn_seed{0..4}` + CIFAR-10 (downloads). The two extra-recipe models (`cifar10_cnn_strongaug`, `cifar10_cnn_noaug_short`) are trained on the first cluster run (~30 min GPU total) and cached as checkpoints, so re-runs reuse them. Outputs: `data/results/nb12_cnn_cifar.json` and `figures/figdata/nb12_error_consistency.{npz,json}` (commit the bundle so the paper figure rebuilds anywhere).

**Caveats.** (1) The 5 seeds share one recipe and can be too similar — the two extra-recipe models are added precisely to widen the spread; check `sim`/`kappa` ranges in §5 before trusting the correlation. (2) This is correlational only. (3) The per-layer factorization uses uniform stimulus weights (each layer factorized independently), so factors at the classifier ≈ classes and deeper factors ≈ sub-classes, matching the paper's purity-drops-toward-input trace. (4) On a laptop the roster is padded with perturbed clones just to exercise the code; those rows are flagged `dummy` and must not be reported.